In [1]:
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import pyarrow as pa

In [2]:
df = pq.read_table("era5_europa_raw_1940_2026.parquet").to_pandas()
df

,valid_time,latitude,longitude,kraj,miasto,temp_C,dewpoint_C,humidity,temp_dew_diff,tp_mm,wind_speed,year,month,day
0,1940-01-01 00:00:00,54.25,18.75,polska,gdansk,-4.613800,-6.333771,87.761589,1.719971,0.000000,5.405678,1940,1,1
1,1940-01-01 01:00:00,54.25,18.75,polska,gdansk,-4.924286,-6.760773,86.951157,1.836487,0.000000,5.169767,1940,1,1
2,1940-01-01 02:00:00,54.25,18.75,polska,gdansk,-4.599396,-6.609314,85.836853,2.009918,0.000000,4.581246,1940,1,1
3,1940-01-01 03:00:00,54.25,18.75,polska,gdansk,-5.009003,-7.707947,81.351677,2.698944,0.000000,4.085026,1940,1,1
4,1940-01-01 04:00:00,54.25,18.75,polska,gdansk,-4.640930,-8.057098,77.011208,3.416168,0.000000,4.043524,1940,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27213403,2026-03-26 19:00:00,40.75,23.00,grecja,thessaloniki,13.080231,6.228882,63.147537,6.851349,0.000477,4.312143,2026,3,26
27213404,2026-03-26 20:00:00,40.75,23.00,grecja,thessaloniki,12.913605,6.248138,63.923645,6.665466,0.022888,4.284801,2026,3,26
27213405,2026-03-26 21:00:00,40.75,23.00,grecja,thessaloniki,12.863007,6.616608,65.782837,6.246399,0.034332,4.021579,2026,3,26
27213406,2026-03-26 22:00:00,40.75,23.00,grecja,thessaloniki,12.714752,7.149628,68.897278,5.565125,0.027180,3.836167,2026,3,26


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27213408 entries, 0 to 27213407
Data columns (total 14 columns):
 #   Column         Dtype         
---  ------         -----         
 0   valid_time     datetime64[ns]
 1   latitude       float32       
 2   longitude      float32       
 3   kraj           object        
 4   miasto         object        
 5   temp_C         float32       
 6   dewpoint_C     float32       
 7   humidity       float32       
 8   temp_dew_diff  float32       
 9   tp_mm          float32       
 10  wind_speed     float32       
 11  year           int16         
 12  month          int8          
 13  day            int8          
dtypes: datetime64[ns](1), float32(8), int16(1), int8(2), object(2)
memory usage: 1.5+ GB


In [4]:
KEY = ["kraj", "miasto"]

In [5]:
def aggregate_extreme_precip(group):
    """
    Klasyfikacja wg natężenia opadu (mm/h):
      słabe:        0.0 - 2.0
      umiarkowane:  2.1 - 5.0
      silne:        5.1 - 10.0
      ulewa:       10.1 - 20.0   ← próg dla tpmm_extreme
      silna ulewa: >= 20.1
    
    Klasyfikacja temperatur (IMGW):
      Upał:        temp_C > 30°C
      Silny mróz:  temp_C <= -15°C

    Kolumny:
      - tpmm_sum          : suma opadów całodobowa [mm]
      - tpmm_extreme      : suma opadów z godzin >= 10.1 mm/h (ulewa+)
      - extreme_hours     : liczba godzin ulewy w dobie
      - heat_hours        : liczba godzin z temperaturą > 30°C (upał)
      - frost_hours       : liczba godzin z temperaturą <= -15°C (silny mróz)

    Źródło: https://imgw.pl/edukacja/slowniki-hydrologiczno-meteorologiczne/
    """
    rain = group["tp_mm"]
    temp = group["temp_C"]
    total = rain.sum()

    PROG_ULEWA  = 10.1   # mm/h
    PROG_UPAL   = 30.0   # °C
    PROG_MROZ   = -15.0  # °C

    if total == 0:
        intense_sum   = 0.0
        intense_hours = 0
    else:
        intense       = rain[rain >= PROG_ULEWA]
        intense_sum   = float(intense.sum())
        intense_hours = int(len(intense))

    return pd.Series({
        "tpmm_sum":      float(total),
        "tpmm_extreme":  intense_sum,
        "extreme_hours": intense_hours,
        "heat_hours":    int((temp > PROG_UPAL).sum()),   # h z upałem w dobie
        "frost_hours":   int((temp <= PROG_MROZ).sum()),  # h z silnym mrozem w dobie
    })

In [6]:
from tqdm import tqdm

cities = df[KEY].drop_duplicates().values.tolist()
n = len(cities)
results = []

for kraj, miasto in tqdm(cities, desc="Agregacja dobowa"):
    chunk = df[(df["kraj"] == kraj) & (df["miasto"] == miasto)]
    grp = chunk.groupby(["year", "month", "day"])

    meteo = (
        grp.agg(
            tempC_mean       = ("temp_C",        "mean"),
            tempC_min        = ("temp_C",        "min"),
            tempC_max        = ("temp_C",        "max"),
            dewpointC_mean   = ("dewpoint_C",    "mean"),
            humidity_mean    = ("humidity",      "mean"),
            tempdewdiff_mean = ("temp_dew_diff", "mean"),
            windspeed_mean   = ("wind_speed",    "mean"),
            windspeed_max    = ("wind_speed",    "max"),
        )
        .reset_index()
    )

    precip = (
        grp.apply(aggregate_extreme_precip, include_groups=False)
        .reset_index()
    )

    daily = meteo.merge(precip, on=["year", "month", "day"])
    daily["kraj"]   = kraj
    daily["miasto"] = miasto
    daily["date"]   = pd.to_datetime(daily[["year", "month", "day"]])
    results.append(daily)

df_daily = pd.concat(results, ignore_index=True)
print(f"   Dobowa: {df_daily.shape}")
print(f"   Kolumny: {list(df_daily.columns)}")

Agregacja dobowa: 100%|████████████████████████████████████████████████████████████████| 36/36 [08:07<00:00, 13.54s/it]

   Dobowa: (1133892, 19)
   Kolumny: ['year', 'month', 'day', 'tempC_mean', 'tempC_min', 'tempC_max', 'dewpointC_mean', 'humidity_mean', 'tempdewdiff_mean', 'windspeed_mean', 'windspeed_max', 'tpmm_sum', 'tpmm_extreme', 'extreme_hours', 'heat_hours', 'frost_hours', 'kraj', 'miasto', 'date']


In [7]:
GROUP_MONTHLY = KEY + ["year", "month"]
results_monthly = []

for kraj, miasto in tqdm(cities, desc="Agregacja miesięczna"):
    chunk = df_daily[(df_daily["kraj"] == kraj) & (df_daily["miasto"] == miasto)]
    
    monthly = (
        chunk.groupby(["year", "month"])
        .agg(
            tempC_mean        = ("tempC_mean",       "mean"),
            tempC_min         = ("tempC_min",         "min"),
            tempC_max         = ("tempC_max",         "max"),
            dewpointC_mean    = ("dewpointC_mean",    "mean"),
            humidity_mean     = ("humidity_mean",     "mean"),
            tempdewdiff_mean  = ("tempdewdiff_mean",  "mean"),
            windspeed_mean    = ("windspeed_mean",    "mean"),
            windspeed_max     = ("windspeed_max",     "max"),
            tpmm_sum          = ("tpmm_sum",          "sum"),
            tpmm_extreme_sum  = ("tpmm_extreme",      "sum"),
            extreme_hours_sum = ("extreme_hours",     "sum"),
            heat_hours_sum    = ("heat_hours",        "sum"),
            frost_hours_sum   = ("frost_hours",       "sum"),
            rainy_days        = ("tpmm_sum",          lambda x: (x > 0.1).sum()),
            heavy_rain_days   = ("tpmm_extreme",      lambda x: (x > 0).sum()),
           
            heat_days         = ("heat_hours",        lambda x: (x > 0).sum()),  # dni z choć 1h upału
            frost_days        = ("frost_hours",       lambda x: (x > 0).sum()),  # dni z choć 1h silnego mrozu
            hot_days          = ("tempC_max",         lambda x: (x > 30).sum()), # dni gdzie max > 30°C
            cold_days         = ("tempC_min",         lambda x: (x <= -15).sum()), # dni gdzie min <= -15°C
        )
        .reset_index()
    )
    monthly["kraj"]  = kraj
    monthly["miasto"] = miasto
    results_monthly.append(monthly)

df_monthly = pd.concat(results_monthly, ignore_index=True)
print(f" Miesięczna gotowa: {df_monthly.shape}")

Agregacja miesięczna: 100%|████████████████████████████████████████████████████████████| 36/36 [00:17<00:00,  2.05it/s]

 Miesięczna gotowa: (37260, 23)


In [8]:
results_yearly = []

for i, (kraj, miasto) in enumerate(cities, 1):
    print(f"[{i}/{n}] {kraj.upper()} – {miasto}...", end=" ")
    
    chunk = df_monthly[(df_monthly["kraj"] == kraj) & (df_monthly["miasto"] == miasto)]
    
    yearly = (
        chunk.groupby(["year"])
        .agg(
            tempC_mean             = ("tempC_mean",          "mean"),
            tempC_min              = ("tempC_min",            "min"),
            tempC_max              = ("tempC_max",            "max"),
            dewpointC_mean         = ("dewpointC_mean",       "mean"),
            humidity_mean          = ("humidity_mean",        "mean"),
            tempdewdiff_mean       = ("tempdewdiff_mean",     "mean"),
            windspeed_mean         = ("windspeed_mean",       "mean"),
            windspeed_max          = ("windspeed_max",        "max"),
            tpmm_annual            = ("tpmm_sum",             "sum"),
            tpmm_extreme_annual    = ("tpmm_extreme_sum",     "sum"),
            extreme_hours_annual   = ("extreme_hours_sum",    "sum"),
            rainy_days_annual      = ("rainy_days",           "sum"),
            heavy_rain_days_annual = ("heavy_rain_days",      "sum"),
           
            heat_hours_annual      = ("heat_hours_sum",       "sum"),  # łączne h upału w roku
            frost_hours_annual     = ("frost_hours_sum",      "sum"),  # łączne h silnego mrozu
            hot_days_annual        = ("hot_days",             "sum"),  # dni z max > 30°C
            cold_days_annual       = ("cold_days",            "sum"),  # dni z min <= -15°C
        )
        .reset_index()
    )
    yearly["kraj"]   = kraj
    yearly["miasto"] = miasto
    results_yearly.append(yearly)
    print(f" {yearly.shape[0]} lat")

df_yearly = pd.concat(results_yearly, ignore_index=True)
print(f"\nRoczna: {df_yearly.shape}")

[1/36] POLSKA – gdansk...  87 lat
[2/36] POLSKA – krakow...  87 lat
[3/36] POLSKA – poznan...  87 lat
[4/36] POLSKA – warszawa...  87 lat
[5/36] FRANCJA – lyon...  87 lat
[6/36] FRANCJA – nice...  87 lat
[7/36] FRANCJA – paris...  87 lat
[8/36] FRANCJA – toulouse...  87 lat
[9/36] HISZPANIA – barcelona...  87 lat
[10/36] HISZPANIA – granada...  87 lat
[11/36] HISZPANIA – madrid...  87 lat
[12/36] HISZPANIA – valencia...  87 lat
[13/36] NORWEGIA – bergen...  87 lat
[14/36] NORWEGIA – oslo...  87 lat
[15/36] NORWEGIA – tromso...  87 lat
[16/36] NORWEGIA – trondheim...  87 lat
[17/36] RUMUNIA – bucharest...  87 lat
[18/36] RUMUNIA – cluj...  87 lat
[19/36] RUMUNIA – iasi...  87 lat
[20/36] RUMUNIA – timisoara...  87 lat
[21/36] TURCJA – ankara...  87 lat
[22/36] TURCJA – antalya...  87 lat
[23/36] TURCJA – istanbul...  87 lat
[24/36] TURCJA – izmir...  87 lat
[25/36] WEGRY – budapest...  87 lat
[26/36] WEGRY – debrecen...  87 lat
[27/36] WEGRY – pecs...  87 lat
[28/36] WEGRY – szeged...  

In [9]:
df_daily.to_parquet("era5europa_daily.parquet",   index=False)
df_monthly.to_parquet("era5europa_monthly.parquet", index=False)
df_yearly.to_parquet("era5europa_yearly.parquet",  index=False)


In [10]:
print(" Statystyki roczne (per miasto):")
stats_yearly = (
    df_yearly.groupby(KEY)
    [["tempC_mean", "tempC_min", "tempC_max",
      "tpmm_annual", "tpmm_extreme_annual",
      "extreme_hours_annual", "heavy_rain_days_annual"]]
    .describe()
    .round(2)
)
print(stats_yearly.head())


 Statystyki roczne (per miasto):
                 tempC_mean                                                  \
                      count   mean   std    min    25%    50%    75%    max   
kraj    miasto                                                                
francja lyon           87.0  11.28  1.06   6.96  10.64  11.18  11.93  13.72   
        nice           87.0  13.27  0.93   9.13  12.76  13.13  13.88  15.47   
        paris          87.0  10.90  0.91   7.75  10.26  10.81  11.50  12.88   
        toulouse       87.0  13.25  0.95   8.70  12.67  13.19  13.85  15.81   
grecja  athens         87.0  17.41  0.92  12.02  16.93  17.44  17.88  20.03   

                 tempC_min        ... extreme_hours_annual       \
                     count  mean  ...                  75%  max   
kraj    miasto                    ...                             
francja lyon          87.0 -9.90  ...                  0.0  2.0   
        nice          87.0 -3.10  ...                  0.0  5.0   

In [11]:

print("1] TOP 10 najcieplejszych dni (tempC_max):")
top_hot = (
    df_daily.nlargest(10, "tempC_max")
    [KEY + ["date", "tempC_max", "tempC_mean"]]
)
print(top_hot.to_string(index=False))


1] TOP 10 najcieplejszych dni (tempC_max):
  kraj    miasto       date  tempC_max  tempC_mean
turcja   antalya 2024-06-13  44.844391   34.774200
grecja    athens 2007-06-26  44.486176   35.303669
turcja   antalya 2000-07-13  44.455597   35.641094
turcja   antalya 1977-07-10  43.772491   33.906895
turcja   antalya 2000-07-07  43.684967   35.353779
turcja   antalya 2000-07-06  43.627960   35.837204
turcja   antalya 1973-07-20  43.541534   34.462902
grecja heraklion 1973-07-20  43.519012   34.523205
grecja    athens 1973-07-19  43.496857   35.000393
turcja   antalya 1956-08-09  43.429169   35.115856


In [12]:
print("\\TOP 10 najbardziej deszczowych dni (tpmm_sum):")
top_rain = (
    df_daily.nlargest(10, "tpmm_sum")
    [KEY + ["date", "tpmm_sum", "tpmm_extreme", "extreme_hours"]]
)
print(top_rain.to_string(index=False))

\TOP 10 najbardziej deszczowych dni (tpmm_sum):
     kraj    miasto       date   tpmm_sum  tpmm_extreme  extreme_hours
   turcja   antalya 1971-12-10 140.299561      0.000000            0.0
   turcja   antalya 2007-01-04 126.744751      0.000000            0.0
  francja      nice 1959-10-20 123.341560     61.272621            5.0
   turcja   antalya 2015-10-23 117.875099     50.058842            4.0
   turcja   antalya 1976-10-15 115.992065     41.826248            3.0
   turcja   antalya 1946-10-18 114.938499     10.641575            1.0
hiszpania  valencia 1982-10-20 107.943054     11.366844            1.0
hiszpania barcelona 1970-10-11 107.134819     10.596752            1.0
  rumunia      iasi 1985-06-18 103.149887      0.000000            0.0
   turcja   antalya 1960-12-20 103.071213     10.129452            1.0


In [13]:
print("TOP 10 dni z najdłuższą ulewa (extreme_hours):")
top_ulewa = (
    df_daily[df_daily["extreme_hours"] > 0]
    .nlargest(10, "extreme_hours")
    [KEY + ["date", "extreme_hours", "tpmm_extreme", "tpmm_sum"]]
)
print(top_ulewa.to_string(index=False))

TOP 10 dni z najdłuższą ulewa (extreme_hours):
   kraj  miasto       date  extreme_hours  tpmm_extreme   tpmm_sum
 wlochy bologna 1966-08-16            7.0     80.046173  90.967651
francja    nice 1959-10-20            5.0     61.272621 123.341560
 turcja antalya 2015-10-23            4.0     50.058842 117.875099
 wlochy bologna 2014-07-30            4.0     44.735909  72.184563
 wlochy   milan 2019-06-10            4.0     44.602871  71.830750
 polska  poznan 2024-07-12            3.0     42.123795  58.870316
rumunia    iasi 1993-09-12            3.0     34.348965  96.409798
 turcja antalya 1976-10-15            3.0     41.826248 115.992065
 wlochy  naples 1959-09-05            3.0     36.208153  56.071758
 wlochy    rome 1977-09-22            3.0     36.632538  66.380501
